In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install structlog -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 929.4 kB/s eta 0:00:00


In [3]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/db/models.py
"""
Database models for storing prediction records
"""

from sqlalchemy import Column, Integer, String, Float, DateTime, Text
from sqlalchemy.ext.declarative import declarative_base
from datetime import datetime

Base = declarative_base()

class PredictionRecord(Base):
    __tablename__ = "predictions"

    id = Column(Integer, primary_key=True, index=True)
    isic_id = Column(String, index=True)
    probability = Column(Float)
    prediction = Column(String)
    model_version = Column(String)
    timestamp = Column(DateTime, default=datetime.utcnow)
    image_filename = Column(String, nullable=True)
    user_feedback = Column(String, nullable=True)



Writing /content/drive/MyDrive/isic-flagship-project/src/db/models.py


In [4]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/repositories/prediction_repo.py
"""
Repository layer for prediction records
"""

from sqlalchemy.orm import Session
from src.db.models import PredictionRecord
from datetime import datetime

class PredictionRepository:
    def __init__(self, db: Session):
        self.db = db

    def save_prediction(self, isic_id: str, probability: float, prediction: str,
                       model_version: str, image_filename: str = None):
        record = PredictionRecord(
            isic_id=isic_id,
            probability=probability,
            prediction=prediction,
            model_version=model_version,
            timestamp=datetime.utcnow(),
            image_filename=image_filename
        )
        self.db.add(record)
        self.db.commit()
        self.db.refresh(record)
        return record



Writing /content/drive/MyDrive/isic-flagship-project/src/repositories/prediction_repo.py


In [5]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/core/logger.py
"""
Structured logging configuration
"""

import structlog


structlog.configure(
    processors=[
        structlog.processors.JSONRenderer()
    ],
    logger_factory=structlog.PrintLoggerFactory(),
)

logger = structlog.get_logger()



Writing /content/drive/MyDrive/isic-flagship-project/src/core/logger.py


In [6]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/main.py
"""
Main FastAPI application
"""

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager

from src.core.config import get_settings
from src.core.logger import logger
from src.api.routes import health_router, prediction_router
from src.api.rag_routes import rag_router

settings = get_settings()

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("Application starting up", app_name=settings.APP_NAME)
    yield
    logger.info("Application shutting down")

app = FastAPI(
    title=settings.APP_NAME,
    version=settings.API_VERSION,
    description="ISIC 2024 Skin Cancer Detection API",
    lifespan=lifespan
)

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

app.include_router(health_router, prefix="/api/v1", tags=["health"])
app.include_router(prediction_router, prefix="/api/v1", tags=["prediction"])
app.include_router(rag_router, prefix="/api/v1", tags=["rag"])

@app.get("/")
async def root():
    logger.info("Root endpoint accessed")
    return {"message": "ISIC 2024 Flagship API is running"}



Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/main.py
